In [8]:
import pandas as pd, numpy as np
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import drive
drive.mount('/content/drive')

WEB = '/content/drive/MyDrive/DEV_project_data/web_data'
df_integrated      = pd.read_csv(f'{WEB}/integrated_cd_year.csv')
df_housing_monthly = pd.read_parquet(f'{WEB}/housing_monthly.parquet')
df_housing_nta     = pd.read_parquet(f'{WEB}/housing_nta.parquet')
df_centroids       = pd.read_csv(f'{WEB}/cd_centroids.csv')
df_crashes         = pd.read_parquet(f'{WEB}/crashes.parquet')

BOROUGH_COLORS = {'Manhattan': '#2557a7', 'Bronx': '#b5562f', 'Brooklyn': '#2f7d5b',
                  'Queens': '#8a4fa8', 'Staten Island': '#c7951c'}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Q6. How does each borough's share of NYC housing problems compare with its share of serious crashes?

**Question:** For each borough, how does its share of all NYC housing maintenance problems compare with its
share of all serious traffic crashes (2022–2025)?

**Why it matters:** if a borough has a bigger share of crashes than of housing problems, its streets carry a
heavier burden than its housing complaints would suggest, and the reverse. This shows where street-safety
and housing-inspection resources would each be most under-matched to the other burden.

**Method:** sum `housing_problems` and `crashes` per borough, convert each to a % of the NYC total, and plot
the gap (crash share − housing share, in percentage points). Positive bars = more crash burden than housing
burden. A grouped bar of the two shares sits beside it.

In [9]:
# ============================================================
# Q6: How does each borough's share of housing problems compare with its share of serious crashes?
# ============================================================
def borough_share_table(df_integrated):
    """Borough totals, each borough's share of the NYC total, and the gap in percentage points."""
    d = df_integrated.groupby('borough', as_index=False)[['housing_problems', 'crashes']].sum()
    d['housing_share'] = d['housing_problems'] / d['housing_problems'].sum() * 100
    d['crash_share']   = d['crashes'] / d['crashes'].sum() * 100
    d['gap_pp']        = d['crash_share'] - d['housing_share']
    d['Borough']       = d['borough'].str.title()
    return d.sort_values('gap_pp').reset_index(drop=True)


def build_q6(df_integrated):
    """Diverging bar of the share gap (left) and the two shares side by side (right)."""
    d = borough_share_table(df_integrated)
    if d.empty:
        return px.bar(title='Q6. No data for the selected filters')

    fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], horizontal_spacing=0.18,
                        subplot_titles=('Gap: crash share minus housing share',
                                        'Share of NYC total, side by side'))
    custom = d[['housing_problems', 'crashes', 'housing_share', 'crash_share']].to_numpy()

    # Left: diverging gap (red = more crash burden than housing burden, blue = the opposite)
    fig.add_trace(go.Bar(
        y=d['Borough'], x=d['gap_pp'], orientation='h', showlegend=False,
        marker_color=np.where(d['gap_pp'] >= 0, '#c0392b', '#2557a7'),
        text=[f'{v:+.1f} pp' for v in d['gap_pp']], textposition='outside', cliponaxis=False,
        customdata=custom,
        hovertemplate=('<b>%{y}</b><br>Gap: %{x:+.1f} pp<br>'
                       'Housing share: %{customdata[2]:.1f}% (%{customdata[0]:,.0f} problems)<br>'
                       'Crash share: %{customdata[3]:.1f}% (%{customdata[1]:,.0f} crashes)<extra></extra>')),
        row=1, col=1)
    fig.add_vline(x=0, line_color='black', line_width=1, row=1, col=1)

    # Right: the two shares as grouped bars
    for col, name, colour in [('housing_share', 'Share of housing problems', '#7f8c8d'),
                              ('crash_share', 'Share of serious crashes', '#c0392b')]:
        fig.add_trace(go.Bar(
            y=d['Borough'], x=d[col], orientation='h', name=name, marker_color=colour,
            text=[f'{v:.1f}%' for v in d[col]], textposition='outside', cliponaxis=False,
            customdata=custom,
            hovertemplate=('<b>%{y}</b><br>' + name + ': %{x:.1f}%<br>'
                           'Housing problems: %{customdata[0]:,.0f}<br>'
                           'Serious crashes: %{customdata[1]:,.0f}<extra></extra>')),
            row=1, col=2)

    gmax = max(abs(d['gap_pp']).max(), 1) * 1.5
    fig.update_xaxes(title_text='Gap (percentage points)', range=[-gmax, gmax], row=1, col=1)
    fig.update_xaxes(title_text='% of NYC total', range=[0, d[['housing_share', 'crash_share']].max().max() * 1.2],
                     row=1, col=2)
    fig.update_yaxes(title_text='Borough', row=1, col=1)
    fig.update_layout(barmode='group', height=450,
                      title='Q6. Borough share of housing problems vs share of serious crashes',
                      legend=dict(orientation='h', y=-0.2, x=0.55))
    return fig


build_q6(df_integrated).show()

In [10]:
# Helper: numbers for the Q6 answer (NOT part of the website)
q6 = borough_share_table(df_integrated)
print(q6[['Borough', 'housing_problems', 'crashes', 'housing_share', 'crash_share', 'gap_pp']].round(1).to_string(index=False))
print('\nTotals: housing', f"{q6['housing_problems'].sum():,}", '| crashes', f"{q6['crashes'].sum():,}")

      Borough  housing_problems  crashes  housing_share  crash_share  gap_pp
        Bronx            912235     2158           34.3         19.9   -14.4
    Manhattan            558768     2172           21.0         20.0    -1.0
     Brooklyn            797155     3361           30.0         31.0     1.0
Staten Island             53391      494            2.0          4.6     2.5
       Queens            336366     2652           12.7         24.5    11.8

Totals: housing 2,657,915 | crashes 10,837


### Q6 Answer

**[Borough A] has the largest crash-burden surplus and [Borough B] the largest deficit.**
[Borough A] holds [X]% of NYC's housing problems but [Y]% of its serious crashes (a gap of +[Z] percentage
points), while [Borough B] holds [X]% of housing problems but only [Y]% of crashes ([Z] pp). [Borough C] is
the closest match between the two shares (gap [Z] pp). Out of [N] serious crashes and [M] housing problems
across NYC in 2022–2025, the shares are therefore **not** proportional, which fits the weak overall association
between the two measures (r = 0.15).

**Benefit to NYC:** boroughs with a positive gap are where street-safety funding is large relative to their
housing complaint load; boroughs with a negative gap have heavy housing-complaint volume that is not matched by
serious crashes, so housing inspection is the larger need there.

**Limitation:** shares use raw counts, not population, road length or building stock, so a borough's share
partly reflects its size. Housing problems are tenant-reported complaints, so they also reflect reporting
behaviour. Only 5 boroughs are compared, so the chart is descriptive and says nothing about causes.

## Q9. Which neighbourhoods have the most multi-problem complaints?

**Question:** Which neighbourhoods (NTAs) have the highest average number of problems per housing complaint
(2022–2025)?

**Why it matters:** one complaint can list several problems (heat, leaks, pests, and so on). A high average
problems-per-complaint is a sign of buildings with several problems at once, which tend to need a full
inspection rather than a single repair.

**Method:** sum `problems` and `complaints` per neighbourhood, divide to get average problems per complaint,
keep only neighbourhoods with at least 500 complaints (to avoid extremes from tiny samples), and show the top 15.

In [11]:
# ============================================================
# Q9: Which neighbourhoods have the most multi-problem complaints?
# ============================================================
MIN_COMPLAINTS = 500

def multi_problem_table(df_housing_nta, min_complaints=MIN_COMPLAINTS):
    """Problems, complaints and average problems per complaint per neighbourhood (NTA)."""
    d = (df_housing_nta.groupby(['nta', 'borough'], as_index=False)[['problems', 'complaints']].sum())
    d = d[d['complaints'] >= min_complaints].copy()
    d['avg_problems_per_complaint'] = d['problems'] / d['complaints']
    d['Borough'] = d['borough'].str.title()
    return d.sort_values('avg_problems_per_complaint', ascending=False).reset_index(drop=True)


def build_q9(df_housing_nta):
    """Horizontal bar of the 15 neighbourhoods with the most problems per complaint."""
    d = multi_problem_table(df_housing_nta).head(15)
    if d.empty:
        return px.bar(title=f'Q9. No neighbourhood has at least {MIN_COMPLAINTS} complaints for the selected filters')

    fig = px.bar(
        d.iloc[::-1], x='avg_problems_per_complaint', y='nta', color='Borough', orientation='h',
        color_discrete_map=BOROUGH_COLORS,
        custom_data=['problems', 'complaints'],
        text=d.iloc[::-1]['avg_problems_per_complaint'].round(2),
        labels={'avg_problems_per_complaint': 'Average problems per complaint',
                'nta': 'Neighbourhood'},
        title=f'Q9. Neighbourhoods with the most problems per complaint (min. {MIN_COMPLAINTS} complaints)')
    fig.update_traces(textposition='outside', cliponaxis=False,
                      hovertemplate=('<b>%{y}</b><br>Avg problems per complaint: %{x:.2f}<br>'
                                     'Problems: %{customdata[0]:,.0f}<br>'
                                     'Complaints: %{customdata[1]:,.0f}<extra></extra>'))
    fig.update_xaxes(range=[0, d['avg_problems_per_complaint'].max() * 1.12])
    fig.update_layout(height=520, yaxis=dict(dtick=1), legend_title_text='Borough')
    return fig


build_q9(df_housing_nta).show()

In [12]:
# Helper: numbers for the Q9 answer (NOT part of the website)
q9_all = multi_problem_table(df_housing_nta)
q9 = q9_all.head(15)
print(f'Neighbourhoods with >= {MIN_COMPLAINTS} complaints: {len(q9_all)}')
print('NYC average problems per complaint:', round(df_housing_nta["problems"].sum() / df_housing_nta["complaints"].sum(), 2))
print(q9[['nta', 'Borough', 'problems', 'complaints', 'avg_problems_per_complaint']].round(2).to_string(index=False))
print('\nBoroughs in the top 15:'); print(q9['Borough'].value_counts().to_string())

Neighbourhoods with >= 500 complaints: 192
NYC average problems per complaint: 1.82
                                      nta       Borough  problems  complaints  avg_problems_per_complaint
           ROSEBANK-SHORE ACRES-PARK HILL Staten Island      9092        3770                        2.41
                            PORT RICHMOND Staten Island      5478        2389                        2.29
                   NEW DORP-MIDLAND BEACH Staten Island      1641         740                        2.22
                  ST. GEORGE-NEW BRIGHTON Staten Island      8628        3962                        2.18
TOMPKINSVILLE-STAPLETON-CLIFTON-FOX HILLS Staten Island      5369        2503                        2.15
                              HUNTS POINT         Bronx     15091        7226                        2.09
             EAST FLATBUSH-REMSEN VILLAGE      Brooklyn     27363       13126                        2.08
  MARINER'S HARBOR-ARLINGTON-GRANITEVILLE Staten Island      5534   

### Q9 Answer

**[NTA 1] has the highest average, with [X.XX] problems per complaint**, followed by [NTA 2] ([X.XX]) and
[NTA 3] ([X.XX]), compared with an NYC-wide average of [X.XX]. Of the top 15, [N] are in [Borough] and [N] in
[Borough]. The gap between the top and the 15th neighbourhood is small ([X.XX] vs [X.XX]), so the ranking
within the top 15 should be read as a group rather than a strict order. Neighbourhoods like these tend to have
complaints that bundle several issues, which is associated with buildings needing broader inspection.

**Benefit to NYC:** HPD could prioritise a full-building inspection (not single-issue repair visits) in these
neighbourhoods.

**Limitation:** one complaint may list several problems simply because of how the tenant or the 311 call-taker
recorded it, so the average partly reflects reporting practice. The 500-complaint minimum removes small
neighbourhoods, and the ranking can change with the year filter.

### AI prompts used (Member 6: Q6 and Q9)

1. "I am member 6" plus the Research Questions Guide PDF and the DataEngineering notebook; asked Claude to complete
   my assigned questions (Q6 borough share gap, Q9 multi-problem neighbourhoods).
2. Claude wrote `build_q6`, `build_q9`, their helper functions and the markdown/answer cells, following the guide's
   rules and the style of the Q1 cells already in the notebook. The bracketed numbers in the answer cells are
   filled in by me from the helper output after running the cells in Colab.